# Round 2 Strategy: ATR Cross-Sectional Alpha

**Pipeline:**
1. Load data (features, universe, returns)
2. Cross-sectional z-score of ATR + clip outliers
3. Construct targets: forward log returns → strip beta → z-score
4. Train PyGAM with monotonic constraint
5. Generate portfolio weights (vol-scaled, dollar-neutral)
6. Backtest using QRT `utils.py`

In [23]:
!pip install seaborn



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import os, sys
import warnings
import yfinance as yf
warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath('phase2_qrt_challenge/scripts'))
from utils import backtest_portfolio

DATA_DIR = 'stores'

# Prediction horizon (trading days)
H = 5

# Rolling beta window (QRT Guideline specifies 250 days)
BETA_WINDOW = 250

# Train/val split ratio
TRAIN_RATIO = 0.70

# Volatility lookback for position sizing
VOL_WINDOW = 20

# Z-score clip threshold
ZSCORE_CLIP = 3.0

# Target GMV in USD for enforcing QRT trading limits (local backtest assumption)
TARGET_GMV = 10_000_000

## 1. Load Data

In [25]:
# Load features (MultiIndex: level 0 = indicator name, level 1 = ticker)
features_df = pd.read_parquet(os.path.join(DATA_DIR, 'features.parquet'))
atr = features_df['average_true_range']  # shape: (dates, tickers)

# Load universe mask (1 = in universe, 0 = out)
universe = pd.read_parquet(os.path.join(DATA_DIR, 'universe_5m.parquet'))

# Load returns
returns = pd.read_parquet(os.path.join(DATA_DIR, 'returns.parquet'))

# Align all three on common dates and tickers
common_dates = atr.index.intersection(universe.index).intersection(returns.index)
common_tickers = atr.columns.intersection(universe.columns).intersection(returns.columns)

atr = atr.loc[common_dates, common_tickers].astype('float64')
universe = universe.loc[common_dates, common_tickers]
returns = returns.loc[common_dates, common_tickers].astype('float64')

print(f'Dates: {len(common_dates)}, Tickers: {len(common_tickers)}')
print(f'Date range: {common_dates[0].date()} → {common_dates[-1].date()}')

Dates: 4092, Tickers: 4999
Date range: 2010-01-04 → 2026-04-10


## 2. Feature Engineering: 4-Factor Model

In [ ]:

# Loading Adj Close and Volume for Feature Engineering
print("Loading price and volume data...")
pv = pd.read_pickle('top_5000_yf_data.pkl')
adj_close = pv['Adj Close'][common_tickers].loc[common_dates].astype('float64')
volume = pv['Volume'][common_tickers].loc[common_dates].astype('float64')

print("Calculating Volatility...")
# 1. Realized Volatility (20 days) -> Cross-sectional Z-score
returns_df = adj_close.pct_change()
vol_20d = returns_df.rolling(20).std()
vol_20d_masked = vol_20d.where(universe == 1)
vol_z = vol_20d_masked.sub(vol_20d_masked.mean(axis=1), axis=0).div(vol_20d_masked.std(axis=1), axis=0)

print("Calculating RSI...")
# 2. RSI (14 days) -> Cross-sectional rank
delta = adj_close.diff()
gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / loss
rsi = 100 - (100 / (1 + rs))
rsi_masked = rsi.where(universe == 1)
rsi_rank = rsi_masked.rank(axis=1, pct=True)

print("Calculating Momentum...")
# 3. Momentum (12M-1M) -> Cross-sectional Z-score
mom_12m_1m = (adj_close.shift(21) / adj_close.shift(252)) - 1
mom_masked = mom_12m_1m.where(universe == 1)
mom_z = mom_masked.sub(mom_masked.mean(axis=1), axis=0).div(mom_masked.std(axis=1), axis=0)

print("Calculating Relative Volume...")
# 4. Relative Volume (today / 20-day avg)
rel_vol = volume / volume.rolling(20).mean()
rel_vol_masked = rel_vol.where(universe == 1)

print("Stacking features...")
stacked_features = pd.concat({
    'volatility': vol_z.stack(dropna=False),
    'rsi': rsi_rank.stack(dropna=False),
    'momentum': mom_z.stack(dropna=False),
    'rel_volume': rel_vol_masked.stack(dropna=False)
}, axis=1)

print(f"Features stacked. Shape: {stacked_features.shape}")


## 3. Target Construction

In [ ]:

# h-day forward log return: ln(P_{t+h} / P_t)
fwd_log_ret = np.log(adj_close.shift(-H) / adj_close)

print('Downloading SPY data for benchmark...')
spy_data = yf.download('SPY', start=common_dates[0], end=common_dates[-1] + pd.Timedelta(days=10), progress=False)

if isinstance(spy_data.columns, pd.MultiIndex):
    if 'Adj Close' in spy_data.columns.levels[0]:
        spy_close = spy_data['Adj Close', 'SPY']
    else:
        spy_close = spy_data['Close', 'SPY']
else:
    spy_close = spy_data.get('Adj Close', spy_data['Close'])

spy_close = spy_close.reindex(common_dates).ffill().squeeze()
market_ret = np.log(spy_close / spy_close.shift(1))

print('Computing rolling betas using 250-day window and QRT formula...')
rolling_var_m = market_ret.rolling(BETA_WINDOW, min_periods=120).var()

returns_masked = returns.where(universe == 1)
betas = pd.DataFrame(index=common_dates, columns=common_tickers, dtype='float64')

for ticker in common_tickers:
    stock_ret = returns_masked[ticker]
    xy = stock_ret * market_ret
    roll_mean_xy = xy.rolling(BETA_WINDOW, min_periods=120).mean()
    roll_mean_x = stock_ret.rolling(BETA_WINDOW, min_periods=120).mean()
    roll_mean_y = market_ret.rolling(BETA_WINDOW, min_periods=120).mean()
    roll_cov = roll_mean_xy - roll_mean_x * roll_mean_y
    raw_beta = roll_cov / rolling_var_m
    betas[ticker] = 0.2 + 0.8 * raw_beta

fwd_market_ret = market_ret.rolling(H).sum().shift(-H)
residual_fwd_ret = fwd_log_ret.sub(betas.mul(fwd_market_ret, axis=0))

residual_fwd_ret_masked = residual_fwd_ret.where(universe == 1)
target_mean = residual_fwd_ret_masked.mean(axis=1)
target_std = residual_fwd_ret_masked.std(axis=1)

Y = residual_fwd_ret_masked.sub(target_mean, axis=0).div(target_std, axis=0)
Y = Y.clip(-ZSCORE_CLIP, ZSCORE_CLIP)

# Stack Target and combine with features
stacked_Y = Y.stack(dropna=False)
stacked_data = stacked_features.copy()
stacked_data['target'] = stacked_Y
print(f"Target stacked. Final combined shape: {stacked_data.shape}")


## 4. Model Training: PyGAM

In [ ]:

from pygam import LinearGAM, s, te
import matplotlib.pyplot as plt

print("Splitting data into Train (2010-2020) and Test (2021+)...")
clean_data = stacked_data.dropna()

train_mask = (clean_data.index.get_level_values(0).year >= 2010) & (clean_data.index.get_level_values(0).year <= 2020)
val_mask = clean_data.index.get_level_values(0).year > 2020

train_data = clean_data[train_mask]
val_data = clean_data[val_mask]

X_train = train_data[['volatility', 'rsi', 'momentum', 'rel_volume']].values
Y_train = train_data['target'].values

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(val_data)}")

gam = LinearGAM(
    s(0, constraints='monotonic_dec', n_splines=10) +  # Volatility (Penalty)
    s(2, constraints='monotonic_inc', n_splines=10) +  # Momentum (Premium)
    te(1, 3, n_splines=(8, 8))                         # RSI x Rel_Volume (Interaction)
)
lam_space = np.logspace(1, 5, 10) 

print("Fitting GAM via Gridsearch (this may take a moment)...")
gam.gridsearch(X_train, Y_train, lam=lam_space, progress=True)

print(f'\nOptimal Lambda chosen: {gam.lam}')
print(f'Pseudo R-Squared: {gam.statistics_["pseudo_r2"]["explained_deviance"]:.6f}')


In [ ]:

# --- GAM Learned Curve and Formulation ---
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Term 0: Volatility
XX = gam.generate_X_grid(term=0)
pdep, confi = gam.partial_dependence(term=0, X=XX, width=0.95)
axs[0].plot(XX[:, 0], pdep, color='darkblue')
axs[0].fill_between(XX[:, 0], confi[:, 0], confi[:, 1], alpha=0.2, color='blue')
axs[0].set_title('Volatility (Monotonic Dec)')
axs[0].axhline(0, color='black', linestyle='--')

# Term 1: Momentum
XX = gam.generate_X_grid(term=2)
pdep, confi = gam.partial_dependence(term=2, X=XX, width=0.95)
axs[1].plot(XX[:, 2], pdep, color='darkred')
axs[1].fill_between(XX[:, 2], confi[:, 0], confi[:, 1], alpha=0.2, color='red')
axs[1].set_title('Momentum (Monotonic Inc)')
axs[1].axhline(0, color='black', linestyle='--')

# Term 2: RSI x Rel_Volume Interaction
XX = gam.generate_X_grid(term=1, n=50)
pdep, _ = gam.partial_dependence(term=1, X=XX)
Z = pdep.reshape(50, 50)
c = axs[2].contourf(XX[:, 1].reshape(50, 50), XX[:, 3].reshape(50, 50), Z, cmap='viridis')
fig.colorbar(c, ax=axs[2])
axs[2].set_title('RSI x Rel_Volume Interaction')
axs[2].set_xlabel('RSI (Ranked)')
axs[2].set_ylabel('Rel Volume')

plt.tight_layout()
plt.show()

# --- Extracting the Mathematical Equation ---
print("--- GAM Mathematical Formulation ---")
coefs = gam.coef_
print(f"Number of coefficients (including intercept): {len(coefs)}")
print(f"Intercept: {coefs[-1]:.6f}")


## 5. Portfolio Construction

In [ ]:

# Re-extract common parameters
VOL_WINDOW = 20
TARGET_GMV = 10_000_000

print('Computing 60-day rolling ADV...')
usd_volume = volume * adj_close
adv_60d = usd_volume.rolling(60, min_periods=30).mean()

def enforce_limits_iterative(alpha_book, limits_book, target_sum=0.5):
    if alpha_book.sum() == 0:
        return pd.Series(0.0, index=alpha_book.index)
    weights = (alpha_book / alpha_book.sum()) * target_sum
    capped = pd.Series(False, index=weights.index)
    for _ in range(20): 
        exceed = weights > limits_book
        if not exceed.any():
            break
        weights[exceed] = limits_book[exceed]
        capped = capped | exceed
        uncapped = ~capped
        if uncapped.sum() == 0:
            break
        remaining_target = target_sum - weights[capped].sum()
        if remaining_target <= 0:
            break
        uncapped_sum = alpha_book[uncapped].sum()
        if uncapped_sum == 0:
            break
        weights[uncapped] = (alpha_book[uncapped] / uncapped_sum) * remaining_target
    return weights

val_dates = val_data.index.get_level_values(0).unique()
portfolio = pd.DataFrame(0.0, index=val_dates, columns=common_tickers)

print('Building rank-based portfolio...')
# Create multi-index X matrix for fast prediction
X_val_all = val_data[['volatility', 'rsi', 'momentum', 'rel_volume']].values
val_data['prediction'] = gam.predict(X_val_all)
pred_df = val_data['prediction'].unstack()

for date in val_dates:
    if date not in pred_df.index: continue
    alpha = pred_df.loc[date].dropna()
    if len(alpha) < 20: continue
    
    # Simple rank based portfolio (do not select quantile, just give weights to all based on rank)
    alpha_rank = alpha.rank()
    alpha_centered = alpha_rank - alpha_rank.mean()
    
    # Separate into long and short sides
    alpha_pos = alpha_centered[alpha_centered > 0]
    alpha_neg = alpha_centered[alpha_centered < 0].abs()
    
    adv_today = adv_60d.loc[date]
    max_usd_from_adv = 0.025 * adv_today[alpha.index]
    max_usd_absolute = 2_000_000
    max_usd_position = np.minimum(max_usd_from_adv, max_usd_absolute)
    
    # Convert USD to weight
    max_weight_limit = np.minimum(0.1, max_usd_position / TARGET_GMV)
    
    limit_pos = max_weight_limit[alpha_pos.index]
    limit_neg = max_weight_limit[alpha_neg.index]
    
    w_pos = enforce_limits_iterative(alpha_pos, limit_pos, target_sum=0.5)
    w_neg = enforce_limits_iterative(alpha_neg, limit_neg, target_sum=0.5)
    
    if w_pos.sum() < 0.49 or w_neg.sum() < 0.49:
        continue
        
    weights = pd.Series(0.0, index=alpha.index)
    weights[w_pos.index] = w_pos
    weights[w_neg.index] = -w_neg
    
    portfolio.loc[date, weights.index] = weights.values

print(f'Portfolio built. Non-zero days: {(portfolio.abs().sum(axis=1) > 0).sum()}')


## 6. Backtest

In [ ]:

bt_dates = val_dates
bt_portfolio = portfolio.loc[bt_dates, common_tickers].fillna(0).astype('float64')
bt_returns = returns.loc[bt_dates, common_tickers].fillna(0).astype('float64')
bt_universe = universe.loc[bt_dates, common_tickers]

active_mask = bt_portfolio.abs().sum(axis=1) > 1e-10
bt_portfolio = bt_portfolio.loc[active_mask]
bt_returns = bt_returns.loc[active_mask]
bt_universe = bt_universe.loc[active_mask]

print(f'Backtest period: {bt_portfolio.index[0].date()} → {bt_portfolio.index[-1].date()}')
print(f'Active trading days: {len(bt_portfolio)}')

net_sharpe, gross_pnl = backtest_portfolio(
    bt_portfolio, bt_returns, bt_universe,
    plot_=True, print_=True
)

cumulative_pnl = gross_pnl.cumsum()
max_drawdown = (cumulative_pnl - cumulative_pnl.cummax()).min()
print(f'Max Drawdown: {max_drawdown:.4f}')
print(f'Final Cumulative PnL: {cumulative_pnl.iloc[-1]:.4f}')
